# Paso 3 del Sprint 1: Generar splits train/val/test

Este notebook hace dos cosas:

1. **Actualiza** `configs/baseline.yaml` con el parametro `class_ratio: 2.0`.
2. **Crea** `scripts/prepare_splits.py`.
3. **Ejecuta** el script para generar `train.csv`, `val.csv`, `test.csv` en `data/processed/`.

**Pre-requisitos:**
- Ya corriste `00_bootstrap_project.ipynb` (estructura creada).
- Ya corriste `00b_download_nih.ipynb` (imagenes NIH en Drive).

**Tiempo:** < 1 minuto total.


## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'


## 2. Actualizar `configs/baseline.yaml`

In [ ]:
from pathlib import Path

yaml_content = r'''# configs/baseline.yaml
# Sprint 1 - baseline binario "No Finding" vs "Cardiomegaly"
# Dataset: NIH ChestX-ray14 (subset images_001 = 4999 imagenes)

project:
  name: "tesis-cxr-baseline"
  sprint: 1
  seed: 42

paths:
  project_root: "/content/drive/MyDrive/Tesis_CXR"
  data_raw: "/content/drive/MyDrive/Tesis_CXR/data/raw/nih"
  data_processed: "/content/drive/MyDrive/Tesis_CXR/data/processed"
  experiments: "/content/drive/MyDrive/Tesis_CXR/experiments"
  checkpoints: "/content/drive/MyDrive/Tesis_CXR/experiments/checkpoints"
  logs: "/content/drive/MyDrive/Tesis_CXR/experiments/logs"

dataset:
  name: "nih"
  csv_filename: "Data_Entry_subset_local.csv"
  images_subdir: "images"
  positive_class: "Cardiomegaly"  # incluye combinadas
  negative_class: "No Finding"
  views: ["PA", "AP"]
  balance_classes: true

# Ratio negativos:positivos tras undersampling
# 2.0 = 2 negativos por cada positivo (balance razonable)
class_ratio: 2.0

preprocessing:
  image_size: 224

splits:
  train_ratio: 0.70
  val_ratio: 0.15
  test_ratio: 0.15
  split_by: "patient_id"

model:
  backbone: "densenet121-res224-nih"
  num_classes: 2
  pretrained: true
  finetune_mode: "head"

training:
  batch_size: 32
  num_workers: 2
  epochs: 10
  learning_rate: 0.001
  weight_decay: 0.0001
  optimizer: "adam"
  patience: 3
  use_amp: true

evaluation:
  metrics: ["auc", "accuracy", "precision", "sensitivity", "specificity"]
  threshold: 0.5

gradcam:
  target_layer: "features.denseblock4"
  num_samples: 6
'''

yaml_path = f'{CODE_DIR}/configs/baseline.yaml'
Path(yaml_path).write_text(yaml_content, encoding='utf-8')
print(f'OK -> {yaml_path}')


## 3. Crear `scripts/prepare_splits.py`

In [ ]:
script_content = r'''"""
scripts/prepare_splits.py

Genera los splits train/val/test para el baseline binario del Sprint 1.

Tarea: clasificacion binaria "No Finding" (0) vs "Cardiomegaly" (1).
Definicion de positivo: imagenes donde 'Cardiomegaly' aparece en Finding Labels
(sola o combinada con otras patologias).

Estrategia:
  1. Leer Data_Entry_subset_local.csv (imagenes que tenemos en disco).
  2. Construir etiqueta binaria.
  3. Filtrar a los casos utiles (positivos + "No Finding"). Descartamos
     los "anormales pero no cardiomegalia" para evitar ambiguedad en el baseline.
  4. Filtrar por vistas PA/AP (frontales), descartar laterales.
  5. Balancear con ratio 2:1 (negativos:positivos) via undersampling.
  6. Split 70/15/15 estratificado POR PACIENTE (no por imagen) para evitar leakage.
  7. Guardar train.csv / val.csv / test.csv en data/processed/.

Uso (desde Colab):
    !python /content/drive/MyDrive/Tesis_CXR/code/scripts/prepare_splits.py

O desde notebook:
    %run /content/drive/MyDrive/Tesis_CXR/code/scripts/prepare_splits.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Permitir imports desde src/ cuando se ejecuta como script
CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.training.utils import set_seed, get_logger, load_config  # noqa: E402


# ---------------------------------------------------------------------------
# Logica de construccion de labels
# ---------------------------------------------------------------------------
def build_binary_labels(df: pd.DataFrame, positive_class: str,
                        negative_class: str) -> pd.DataFrame:
    """
    Construye una columna 'label' binaria:
      - 1 si 'positive_class' aparece en 'Finding Labels' (sola o combinada).
      - 0 si 'Finding Labels' == 'negative_class' exactamente.
      - NaN si no cae en ninguno de los dos (se filtra despues).

    Se descartan los "anormales pero sin cardiomegalia" para que la clase 0
    represente inequivocamente 'sano' y la clase 1 represente 'cardiomegalia'.
    """
    df = df.copy()

    # Positivo: contiene la palabra exacta (ej: 'Cardiomegaly|Effusion' cuenta)
    is_positive = df['Finding Labels'].str.contains(positive_class, regex=False)
    # Negativo: exactamente 'No Finding' (no combina con nada)
    is_negative = df['Finding Labels'] == negative_class

    df['label'] = np.where(is_positive, 1,
                   np.where(is_negative, 0, np.nan))
    return df


def filter_views(df: pd.DataFrame, views: list[str]) -> pd.DataFrame:
    """Deja solo las vistas aceptadas (tipicamente PA y AP frontales)."""
    return df[df['View Position'].isin(views)].copy()


def undersample_negatives(df: pd.DataFrame, ratio: float,
                          seed: int) -> pd.DataFrame:
    """
    Submuestrea la clase mayoritaria (negativa) para alcanzar el ratio deseado.
    ratio = n_negativos / n_positivos.

    Ejemplo: 196 positivos, ratio=2.0 -> 392 negativos.
    """
    positives = df[df['label'] == 1]
    negatives = df[df['label'] == 0]

    n_pos = len(positives)
    n_neg_target = min(len(negatives), int(n_pos * ratio))

    negatives_sampled = negatives.sample(n=n_neg_target, random_state=seed)
    balanced = pd.concat([positives, negatives_sampled], ignore_index=True)
    return balanced.sample(frac=1.0, random_state=seed).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Split por paciente (critico para evitar data leakage)
# ---------------------------------------------------------------------------
def split_by_patient(df: pd.DataFrame, train_ratio: float, val_ratio: float,
                     test_ratio: float, seed: int
                     ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Divide el DataFrame en train/val/test agrupando por 'Patient ID'.
    Todas las imagenes del mismo paciente caen en el MISMO split.

    Esto es CRITICO para que las metricas del test sean honestas:
    si el paciente X aparece en train y test, el modelo "memoriza" al paciente
    y las metricas quedan infladas. En una tesis eso es inaceptable.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "Los ratios deben sumar 1.0"

    rng = np.random.default_rng(seed)

    # Obtener pacientes unicos y mezclarlos
    unique_patients = df['Patient ID'].unique()
    rng.shuffle(unique_patients)

    n = len(unique_patients)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    # test se lleva el resto (evita perdida por redondeo)

    train_patients = set(unique_patients[:n_train])
    val_patients = set(unique_patients[n_train:n_train + n_val])
    test_patients = set(unique_patients[n_train + n_val:])

    train_df = df[df['Patient ID'].isin(train_patients)].reset_index(drop=True)
    val_df = df[df['Patient ID'].isin(val_patients)].reset_index(drop=True)
    test_df = df[df['Patient ID'].isin(test_patients)].reset_index(drop=True)

    return train_df, val_df, test_df


# ---------------------------------------------------------------------------
# Reporte
# ---------------------------------------------------------------------------
def report_split(name: str, df: pd.DataFrame, logger) -> None:
    """Imprime estadisticas de un split."""
    n = len(df)
    n_pos = int((df['label'] == 1).sum())
    n_neg = int((df['label'] == 0).sum())
    n_patients = df['Patient ID'].nunique()
    pct_pos = (n_pos / n * 100) if n else 0.0

    logger.info(
        "%s: %4d imagenes | %3d pacientes | "
        "positivos=%3d (%.1f%%) | negativos=%3d",
        name.upper(), n, n_patients, n_pos, pct_pos, n_neg
    )


def check_no_leakage(train: pd.DataFrame, val: pd.DataFrame,
                     test: pd.DataFrame, logger) -> None:
    """Valida que ningun paciente este en mas de un split."""
    p_train = set(train['Patient ID'])
    p_val = set(val['Patient ID'])
    p_test = set(test['Patient ID'])

    overlaps = {
        'train∩val': p_train & p_val,
        'train∩test': p_train & p_test,
        'val∩test': p_val & p_test,
    }

    for name, overlap in overlaps.items():
        if overlap:
            logger.error("LEAKAGE en %s: %d pacientes compartidos", name, len(overlap))
            raise AssertionError(f"Data leakage detectado en {name}")

    logger.info("Sin leakage: pacientes disjuntos entre splits.")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main(config_path: str | None = None) -> None:
    # Cargar config
    if config_path is None:
        config_path = CODE_DIR / "configs" / "baseline.yaml"
    cfg = load_config(config_path)

    # Seed y logger
    set_seed(cfg.project['seed'])
    log_file = Path(cfg.paths['logs']) / "prepare_splits.log"
    logger = get_logger("prepare_splits", log_file=log_file)

    logger.info("=" * 60)
    logger.info("PREPARE SPLITS - Sprint 1 baseline")
    logger.info("=" * 60)

    # Rutas
    nih_dir = Path(cfg.paths['data_raw'])
    processed_dir = Path(cfg.paths['data_processed'])
    processed_dir.mkdir(parents=True, exist_ok=True)

    subset_csv = nih_dir / "Data_Entry_subset_local.csv"
    if not subset_csv.exists():
        raise FileNotFoundError(
            f"No se encontro {subset_csv}. "
            "Corre primero el notebook 00b_download_nih.ipynb."
        )

    # 1. Cargar
    df = pd.read_csv(subset_csv)
    logger.info("CSV cargado: %d filas", len(df))

    # 2. Filtrar por vistas
    df = filter_views(df, cfg.dataset['views'])
    logger.info("Tras filtrar vistas %s: %d filas", cfg.dataset['views'], len(df))

    # 3. Construir labels binarios
    df = build_binary_labels(
        df,
        positive_class=cfg.dataset['positive_class'],
        negative_class=cfg.dataset['negative_class'],
    )
    n_before = len(df)
    df = df.dropna(subset=['label']).copy()
    df['label'] = df['label'].astype(int)
    logger.info(
        "Tras construir labels: %d filas (descartadas %d ambiguas)",
        len(df), n_before - len(df),
    )
    logger.info(
        "  Positivos (%s): %d | Negativos (%s): %d",
        cfg.dataset['positive_class'], int((df['label'] == 1).sum()),
        cfg.dataset['negative_class'], int((df['label'] == 0).sum()),
    )

    # 4. Balancear con ratio 2:1
    ratio = cfg.get('class_ratio', 2.0)
    df_balanced = undersample_negatives(df, ratio=ratio, seed=cfg.project['seed'])
    logger.info("Tras balancear (ratio %.1f:1): %d filas", ratio, len(df_balanced))

    # 5. Split por paciente
    train_df, val_df, test_df = split_by_patient(
        df_balanced,
        train_ratio=cfg.splits['train_ratio'],
        val_ratio=cfg.splits['val_ratio'],
        test_ratio=cfg.splits['test_ratio'],
        seed=cfg.project['seed'],
    )

    # 6. Reportar y validar
    logger.info("-" * 60)
    report_split("train", train_df, logger)
    report_split("val", val_df, logger)
    report_split("test", test_df, logger)
    logger.info("-" * 60)
    check_no_leakage(train_df, val_df, test_df, logger)

    # 7. Guardar
    columns_to_save = ['Image Index', 'Finding Labels', 'Patient ID',
                       'View Position', 'label']
    train_df[columns_to_save].to_csv(processed_dir / "train.csv", index=False)
    val_df[columns_to_save].to_csv(processed_dir / "val.csv", index=False)
    test_df[columns_to_save].to_csv(processed_dir / "test.csv", index=False)

    logger.info("Splits guardados en: %s", processed_dir)
    logger.info("  train.csv: %d filas", len(train_df))
    logger.info("  val.csv:   %d filas", len(val_df))
    logger.info("  test.csv:  %d filas", len(test_df))
    logger.info("=" * 60)
    logger.info("PREPARE SPLITS OK")


if __name__ == "__main__":
    main()
'''

script_path = f'{CODE_DIR}/scripts/prepare_splits.py'
Path(script_path).write_text(script_content, encoding='utf-8')
print(f'OK -> {script_path}')


## 4. Ejecutar el script

Si todo va bien, veras un reporte con la distribucion de cada split y el tiempo total (~1 segundo).

In [ ]:
import os
NIH_DIR = f'{PROJECT_ROOT}/data/raw/nih'
print(f'Contenido de {NIH_DIR}:')
for f in sorted(os.listdir(NIH_DIR)):
    size_mb = os.path.getsize(f'{NIH_DIR}/{f}') / (1024**2) if os.path.isfile(f'{NIH_DIR}/{f}') else 0
    print(f'  {f:40s} {size_mb:>8.2f} MB' if size_mb else f'  {f}/ (directorio)')

In [ ]:
!python $CODE_DIR/scripts/prepare_splits.py


In [ ]:
# Alternativa si el comando anterior no expande la variable:
import subprocess
result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/prepare_splits.py'],
    capture_output=True, text=True
)
print('STDOUT:')
print(result.stdout)
print('STDERR:')
print(result.stderr)


## 5. Verificar los CSVs generados

In [ ]:
import pandas as pd

processed = f'{PROJECT_ROOT}/data/processed'

for name in ['train', 'val', 'test']:
    df = pd.read_csv(f'{processed}/{name}.csv')
    n_pos = (df['label'] == 1).sum()
    n_neg = (df['label'] == 0).sum()
    n_pat = df['Patient ID'].nunique()
    print(f'{name:5s}: {len(df):4d} imgs | {n_pat:3d} pacientes | '
          f'pos={n_pos:3d} | neg={n_neg:3d}')

print()
print('Primeras 3 filas de train.csv:')
pd.read_csv(f'{processed}/train.csv').head(3)


## 6. Siguiente paso

Si los splits se generaron correctamente, vuelve al chat para recibir:

1. `src/datasets/transforms.py` - preprocesamiento de imagenes (rescate de tu notebook viejo).
2. `src/datasets/nih.py` - Dataset de PyTorch que lee los CSVs y carga imagenes.
3. Notebook de EDA para visualizar ejemplos antes de entrenar.
